# Statistical Analysis

This notebook performs correlation analysis and identifies at-risk LSOAs.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import os

OUTPUT_FOLDER = 'output'
INPUT_FILE = os.path.join(OUTPUT_FOLDER, 'Leeds_IMD_with_Health.csv')

print("=" * 60)
print("STATISTICAL ANALYSIS")
print("=" * 60)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Data file not found: {INPUT_FILE}. Please run 03_Health_Outcomes.ipynb first.")

print(f"\nLoading data from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows")

STATISTICAL ANALYSIS

Loading data from output\Leeds_IMD_with_Health.csv...
Loaded 482 rows


## Correlation Analysis

In [2]:
print("\n" + "=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)

# Find deprivation score columns
deprivation_cols = []
for col in df.columns:
    if 'score' in col.lower() and df[col].dtype in [np.float64, np.int64]:
        deprivation_cols.append(col)

# Find health indicator columns
health_cols = [col for col in df.columns 
               if any(term in col.lower() for term in 
                     ['obesity', 'diabetes', 'life_expectancy', 'mental_health',
                      'smoking', 'physical_activity', 'hospital', 'health_risk'])]

if not deprivation_cols or not health_cols:
    print("Warning: Could not find sufficient columns for correlation analysis")
else:
    # Calculate correlations
    correlations = {}
    
    print("\nCorrelations between Deprivation and Health Indicators:")
    print("-" * 60)
    
    for health_col in health_cols:
        health_data = df[health_col].dropna()
        
        for depriv_col in deprivation_cols:
            depriv_data = df[depriv_col].dropna()
            
            # Align indices
            common_idx = health_data.index.intersection(depriv_data.index)
            if len(common_idx) > 10:
                corr = df.loc[common_idx, health_col].corr(df.loc[common_idx, depriv_col])
                
                if not pd.isna(corr):
                    key = f"{depriv_col} vs {health_col}"
                    correlations[key] = corr
                    
                    print(f"{depriv_col[:30]:30s} vs {health_col[:30]:30s}: {corr:6.3f}")
    
    # Create correlation dataframe
    if correlations:
        corr_df = pd.DataFrame(list(correlations.items()), 
                              columns=['Variable Pair', 'Correlation'])
        corr_df = corr_df.sort_values('Correlation', key=abs, ascending=False)
        
        print("\nTop correlations (by absolute value):")
        print(corr_df.head(10).to_string(index=False))


CORRELATION ANALYSIS

Correlations between Deprivation and Health Indicators:
------------------------------------------------------------
Index of Multiple Deprivation  vs Obesity_Percent               :  0.886
Income Score (rate)            vs Obesity_Percent               :  0.864
Employment Score (rate)        vs Obesity_Percent               :  0.839
Education, Skills and Training vs Obesity_Percent               :  0.838
Health Deprivation and Disabil vs Obesity_Percent               :  0.783
Crime Score                    vs Obesity_Percent               :  0.754
Barriers to Housing and Servic vs Obesity_Percent               :  0.164
Living Environment Score       vs Obesity_Percent               :  0.472
Income Deprivation Affecting C vs Obesity_Percent               :  0.826
Income Deprivation Affecting O vs Obesity_Percent               :  0.747
Children and Young People Sub- vs Obesity_Percent               :  0.774
Adult Skills Sub-domain Score  vs Obesity_Percent        

## Identify At-Risk LSOAs

In [3]:
print("\n" + "=" * 60)
print("IDENTIFYING TOP 10 AT-RISK LSOAs")
print("=" * 60)

# Find IMD score column
imd_score_col = None
for col in df.columns:
    if 'imd' in col.lower() and 'score' in col.lower():
        imd_score_col = col
        break

# Find health risk score
health_risk_col = 'Health_Risk_Score' if 'Health_Risk_Score' in df.columns else None

if not imd_score_col or not health_risk_col:
    print("Warning: Could not find required columns for risk identification")
else:
    # Normalize both scores to 0-1 scale
    imd_values = df[imd_score_col].fillna(df[imd_score_col].median())
    imd_min = imd_values.min()
    imd_max = imd_values.max()
    imd_normalized = (imd_values - imd_min) / (imd_max - imd_min) if imd_max > imd_min else imd_values
    
    health_values = df[health_risk_col].fillna(df[health_risk_col].median())
    health_min = health_values.min()
    health_max = health_values.max()
    health_normalized = (health_values - health_min) / (health_max - health_min) if health_max > health_min else health_values
    
    # Create composite risk score
    df['Composite_Risk_Score'] = (imd_normalized + health_normalized) / 2
    
    # Find LSOA identifier column
    lsoa_id_col = None
    for col in df.columns:
        if 'lsoa' in col.lower() and ('code' in col.lower() or 'name' in col.lower()):
            lsoa_id_col = col
            break
    
    # Select top at-risk LSOAs
    at_risk = df.nlargest(10, 'Composite_Risk_Score').copy()
    
    # Prepare output
    output_cols = []
    if lsoa_id_col:
        output_cols.append(lsoa_id_col)
    output_cols.extend([imd_score_col, health_risk_col, 'Composite_Risk_Score'])
    
    # Add health indicators
    health_indicator_cols = [col for col in df.columns 
                            if any(term in col.lower() for term in 
                                  ['obesity', 'diabetes', 'life_expectancy', 'smoking'])]
    output_cols.extend(health_indicator_cols[:5])
    
    available_cols = [col for col in output_cols if col in at_risk.columns]
    at_risk_output = at_risk[available_cols].copy()
    
    print(f"\nTop 10 At-Risk LSOAs:")
    print("-" * 60)
    print(at_risk_output.to_string(index=False))
    
    # Save to file
    output_path = os.path.join(OUTPUT_FOLDER, 'top_at_risk_lsoas.csv')
    at_risk_output.to_csv(output_path, index=False)
    print(f"\nSaved to {output_path}")


IDENTIFYING TOP 10 AT-RISK LSOAs

Top 10 At-Risk LSOAs:
------------------------------------------------------------
LSOA code (2011)  Index of Multiple Deprivation (IMD) Score  Health_Risk_Score  Composite_Risk_Score  Obesity_Percent  Diabetes_Percent  Life_Expectancy  Smoking_Percent
       E01011372                                     78.577           0.891342              0.995080        34.158709         14.270693        77.440165        24.016728
       E01011368                                     75.543           0.898983              0.980184        34.927636         12.866333        76.734255        24.773225
       E01011363                                     73.670           0.847002              0.934479        33.254318         11.786931        78.528191        26.759852
       E01011662                                     77.034           0.797084              0.924306        34.746689         11.221525        78.154956        25.998569
       E01011375                

## Build Predictive Model

In [4]:
print("\n" + "=" * 60)
print("PREDICTIVE MODELING")
print("=" * 60)

# Find domain score columns
domain_names = ['Income', 'Employment', 'Education', 'Health', 'Crime', 'Housing', 'Environment']
domain_cols = []

for domain in domain_names:
    matching_cols = [col for col in df.columns 
                    if domain.lower() in col.lower() and 'score' in col.lower()]
    if matching_cols:
        domain_cols.append(matching_cols[0])

# Find target health indicator
target_col = 'Health_Risk_Score' if 'Health_Risk_Score' in df.columns else None

if not domain_cols or not target_col:
    print("Warning: Insufficient data for predictive modeling")
else:
    # Prepare data
    feature_cols = domain_cols
    data = df[feature_cols + [target_col]].dropna()
    
    if len(data) < 10:
        print("Warning: Insufficient data points for modeling")
    else:
        X = data[feature_cols]
        y = data[target_col]
        
        # Build model
        model = LinearRegression()
        model.fit(X, y)
        
        # Predictions
        y_pred = model.predict(X)
        r2 = r2_score(y, y_pred)
        
        # Feature importance (coefficients)
        feature_importance = pd.DataFrame({
            'Feature': feature_cols,
            'Coefficient': model.coef_,
            'Abs_Coefficient': np.abs(model.coef_)
        }).sort_values('Abs_Coefficient', ascending=False)
        
        print(f"\nModel Performance:")
        print(f"  Target Variable: {target_col}")
        print(f"  R² Score: {r2:.4f}")
        print(f"  Number of samples: {len(data)}")
        
        print(f"\nFeature Importance (Coefficients):")
        print(feature_importance.to_string(index=False))
        
        # Save results
        output_path = os.path.join(OUTPUT_FOLDER, 'model_results.csv')
        feature_importance.to_csv(output_path, index=False)
        print(f"\nModel results saved to {output_path}")

print("\n" + "=" * 60)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 60)


PREDICTIVE MODELING

Model Performance:
  Target Variable: Health_Risk_Score
  R² Score: 0.9551
  Number of samples: 482

Feature Importance (Coefficients):
                                Feature  Coefficient  Abs_Coefficient
                Employment Score (rate)     0.599535         0.599535
                    Income Score (rate)     0.541602         0.541602
                            Crime Score     0.024866         0.024866
Health Deprivation and Disability Score     0.019567         0.019567
   Education, Skills and Training Score     0.001326         0.001326
 Barriers to Housing and Services Score     0.001305         0.001305
               Living Environment Score     0.001227         0.001227

Model results saved to output\model_results.csv

STATISTICAL ANALYSIS COMPLETE
